### 📈 Quantitative Finance in 3 Minutes

##### ▶️ Related Quant Guild Videos:

- [Time Series Analysis for Quant Finance](https://youtu.be/JwqjuUnR8OY)

- [Quant Trader on Retail vs Institutional Trading](https://youtu.be/j1XAcdEHzbU)

- [Quant on Trading and Investing](https://youtu.be/CKXp_sMwPuY)

- [Why Poker Pros Make the Best Traders (It's NOT Luck)](https://youtu.be/wZChBKDFFeU)

- [Quant vs. Discretionary Trading](https://youtu.be/3gblERSSHXI)

- [Quant Busts 3 Trading Myths with Math](https://youtu.be/wJfIk3VnubE)

###### ______________________________________________________________________________________________________________________________________

##### [🚀 Master your Quantitative Skills with Quant Guild](https://quantguild.com)

##### [📚 Visit the Quant Guild Library for more Jupyter Notebooks](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [📈 Interactive Brokers for Algorithmic Trading](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

##### [👾 Join the Quant Guild Discord Server](discord.com/invite/MJ4FU2c6c3)

---

In [ ]:
%%html
<style>
/* Overwrite the hard-coded white background for ipywidgets */
.cell-output-ipywidget-background {
    background-color: transparent !important;
}
/* Set widget foreground text and color to match the VS Code dark theme */
:root {
    --jp-widgets-color: var(--vscode-editor-foreground);
    --jp-widgets-font-size: var(--vscode-editor-font-size);
}
</style>

### 📖 Sections

#### 1.) 📈 How Much Do I Pay for This?

- Roullette vs. European Options

- Accumulating an edge making a market

#### 2.) 🎯 What is the Optimal Decision?

- AAPL vs. NVDA

- Accumulating an edge making optimal decisions

#### 3.) 💭 Closing Thoughts and Future Topics

---

#### 1.) 📈 How Much Do I Pay for This?

Consider these two games
- Roullette
- European Option

What is the fair price of these zero-sum games such that on average wealth doesn't drift?

Statistically, all we need is the expected value (mean, or average)...

###### ______________________________________________________________________________________________________________________________________

##### 🎰 Roullette is Easy

Physically, Roullette is a game of chance, asymptotically all distributions, probabilities, and statistics converge

 The expected value for betting \$1 on red in American Roulette is:
 $$
     \mathbb{E}[\text{Payout}] = \frac{18}{38} \times 1 + \frac{20}{38} \times (-1) = -\frac{2}{38} \approx -0.0526
 $$

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# --- 1. DATA GENERATION & MATH SETUP ---
# ==========================================
np.random.seed(42)

N_spins = 3000  # Increased to 10,000
n_players = 5
start_wealth = 50

# Casino Math (American Roulette betting on Red)
# 18 Red (+1), 20 Black/Green (-1)
p_win = 18 / 38
p_lose = 20 / 38
theoretical_edge = (18 * 1 + 20 * -1) / 38  # Approx -0.0526

t_steps = np.arange(0, N_spins + 1)

# A. Fair Game Paths (EV = 0)
fair_steps = np.random.choice([1, -1], size=(n_players, N_spins))
fair_paths = np.zeros((n_players, N_spins + 1))
fair_paths[:, 0] = start_wealth
for i in range(n_players):
    fair_paths[i, 1:] = start_wealth + np.cumsum(fair_steps[i])

# B. Actual Casino Paths (EV < 0 with absorbing state at 0)
actual_steps = np.random.choice([1, -1], p=[p_win, p_lose], size=(n_players, N_spins))
actual_paths = np.zeros((n_players, N_spins + 1))
actual_paths[:, 0] = start_wealth
for i in range(n_players):
    for t in range(1, N_spins + 1):
        if actual_paths[i, t - 1] <= 0:
            actual_paths[i, t] = 0  # Absorbed
        else:
            actual_paths[i, t] = actual_paths[i, t - 1] + actual_steps[i, t - 1]

# C. Global Spin Sequence (For Bar Chart & LLN)
# 0-17: Red, 18-35: Black, 36-37: Green
global_spins = np.random.randint(0, 38, size=N_spins)
spin_values = np.where(global_spins < 18, 1, -1)

# Law of Large Numbers Array
lln = np.zeros(N_spins + 1)
lln[1:] = np.cumsum(spin_values) / np.arange(1, N_spins + 1)

# Bar chart settings
bar_x = ['Red', 'Black', 'Green']
bar_y = [18/38, 18/38, 2/38]

def get_bar_colors(spin_val):
    # Dimmed colors by default (Red, Grayish-Black for dark theme visibility, Green)
    colors = ['rgba(255, 0, 0, 0.15)', 'rgba(150, 150, 150, 0.15)', 'rgba(0, 255, 0, 0.15)']
    if spin_val is None:
        return colors
    if spin_val < 18:
        colors[0] = 'rgba(255, 0, 0, 1.0)'      # Active Red
    elif spin_val < 36:
        colors[1] = 'rgba(200, 200, 200, 1.0)'  # Active Black (Light Gray to pop on dark bg)
    else:
        colors[2] = 'rgba(0, 255, 0, 1.0)'      # Active Green
    return colors

# Path Colors (Varying Opacities)
# Opacities range from 0.4 to 1.0
fair_colors = [f'rgba(255, 255, 255, {0.4 + 0.15 * i})' for i in range(n_players)]
actual_colors = [f'rgba(255, 0, 0, {0.4 + 0.15 * i})' for i in range(n_players)]

# ==========================================
# --- 2. FIGURE & TRACE INITIALIZATION ---
# ==========================================
fig = make_subplots(
    rows=2, cols=2, 
    vertical_spacing=0.15, horizontal_spacing=0.10,
    subplot_titles=[
        "<b>Fair Game Wealth (EV = 0)</b>", 
        "<b>Current Spin Probability & Draw</b>", 
        "<b>Actual Roulette Wealth (EV < 0)</b>", 
        "<b>Law of Large Numbers Convergence</b>"
    ]
)

# 0..4: Top-Left Traces (Fair Paths - White)
for i in range(n_players):
    fig.add_trace(go.Scatter(x=[0], y=[start_wealth], mode='lines', line=dict(color=fair_colors[i], width=2)), row=1, col=1)

# 5: Top-Right Trace (Bar Chart)
fig.add_trace(go.Bar(x=bar_x, y=bar_y, marker_color=get_bar_colors(None)), row=1, col=2)

# 6..10: Bottom-Left Traces (Actual Paths - Red)
for i in range(n_players):
    fig.add_trace(go.Scatter(x=[0], y=[start_wealth], mode='lines', line=dict(color=actual_colors[i], width=2)), row=2, col=1)

# 11: Bottom-Right Trace (LLN)
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines', line=dict(color='#00ffff', width=2)), row=2, col=2)

# Static H-Lines for Law of Large Numbers chart
# Theoretical casino edge (Negative)
fig.add_shape(type="line", x0=0, y0=theoretical_edge, x1=N_spins, y1=theoretical_edge, 
              line=dict(color="red", width=2, dash="dash"), row=2, col=2)
# Fair value (Zero)
fig.add_shape(type="line", x0=0, y0=0, x1=N_spins, y1=0, 
              line=dict(color="white", width=2, dash="dot"), row=2, col=2)

# ==========================================
# --- 3. ANIMATION FRAMES & SLIDER ---
# ==========================================
frames = []
step_size = max(1, N_spins // 100) # Ensure ~100 frames for smooth playback
frame_indices = list(range(1, N_spins + 1, step_size))
if frame_indices[-1] != N_spins:
    frame_indices.append(N_spins)

slider_steps = []

for k in frame_indices:
    frame_data = []
    
    # Update TL (Fair Paths)
    for i in range(n_players):
        frame_data.append(go.Scatter(x=t_steps[:k+1], y=fair_paths[i, :k+1]))
        
    # Update TR (Active Draw highlight)
    current_spin = global_spins[k-1]
    frame_data.append(go.Bar(marker_color=get_bar_colors(current_spin)))
    
    # Update BL (Actual Paths)
    for i in range(n_players):
        frame_data.append(go.Scatter(x=t_steps[:k+1], y=actual_paths[i, :k+1]))
        
    # Update BR (LLN)
    frame_data.append(go.Scatter(x=t_steps[1:k+1], y=lln[1:k+1]))
    
    frame_name = f"step{k}"
    frames.append(go.Frame(data=frame_data, name=frame_name))
    
    slider_steps.append({
        "args": [[frame_name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate", "transition": {"duration": 0}}],
        "label": f"{k}",
        "method": "animate"
    })

fig.frames = frames

# ==========================================
# --- 4. LAYOUT & ANNOTATIONS ---
# ==========================================
fig.update_layout(
    height=850, width=1100,
    template="plotly_dark",
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    showlegend=False,
    sliders=[{
        "active": 0,
        "yanchor": "top", "xanchor": "left",
        "currentvalue": {"font": {"size": 14, "color": "white"}, "prefix": "Spin Number: ", "visible": True, "xanchor": "right"},
        "pad": {"b": 10, "t": 60},
        "len": 0.82, "x": 0.18, "y": -0.10,
        "steps": slider_steps
    }],
    updatemenus=[{
        "type": "buttons",
        "showactive": False,
        "x": 0.0, "y": -0.10, 
        "xanchor": "left", "yanchor": "top",
        "direction": "left",
        "bgcolor": "rgba(0,0,0,0)",
        "font": {"color": "white", "size": 12}, 
        "buttons": [{
            "label": "▶ Play Simulation",
            "method": "animate",
            "args": [None, {"frame": {"duration": 40, "redraw": True}, "fromcurrent": True, "mode": "immediate"}]
        }],
        "bordercolor": "white", 
        "borderwidth": 1
    }]
)

# Set strictly bounded axes so the animation stays stable
y_max_fair = np.max(fair_paths) * 1.05
y_min_fair = min(np.min(fair_paths) * 1.05, -10) # Added buffer for negative dips in fair game

fig.update_xaxes(title_text='Spins', range=[0, N_spins], row=1, col=1)
fig.update_yaxes(title_text='Wealth ($)', range=[y_min_fair, y_max_fair], row=1, col=1)

fig.update_yaxes(title_text='Probability', range=[0, 0.6], tickformat=".0%", row=1, col=2)

fig.update_xaxes(title_text='Spins', range=[0, N_spins], row=2, col=1)
fig.update_yaxes(title_text='Wealth ($)', range=[0, start_wealth * 1.5], row=2, col=1)

fig.update_xaxes(title_text='Spins', range=[0, N_spins], row=2, col=2)
fig.update_yaxes(title_text='Cumulative Mean', range=[-0.4, 0.4], row=2, col=2)

# Information Annotations
fig.add_annotation(text="<b>Zero Edge</b><br>Paths oscillate randomly", row=1, col=1, x=0.02, y=0.95, xref="x domain", yref="y domain", showarrow=False, font=dict(color="white"), bgcolor="rgba(0,0,0,0.6)", bordercolor="white", borderwidth=1)
fig.add_annotation(text="<b>Negative Edge</b><br>Paths hit absorbing state at $0", row=2, col=1, x=0.02, y=0.95, xref="x domain", yref="y domain", showarrow=False, font=dict(color="red"), bgcolor="rgba(0,0,0,0.6)", bordercolor="red", borderwidth=1)
fig.add_annotation(text="<b>Expected Value</b><br>-5.26% per spin", row=2, col=2, x=0.98, y=0.95, xref="x domain", yref="y domain", showarrow=False, font=dict(color="red"), bgcolor="rgba(0,0,0,0.6)", bordercolor="red", borderwidth=1)

fig.show()

###### ______________________________________________________________________________________________________________________________________

##### 📜 European Options are Harder

Options derive their value on stock, there is no asymptotic convergence so we can't just observe a physical expectation

However, Black-Scholes proposes a hedging argument for the risk-neutral (hedged portfolio) expected value of the contract

 $$
 C = e^{-rT} \mathbb{E}^{\mathbb{Q}} \left[ \max(S_T - K, 0) \right]
 $$

In other words, if we perfectly hedge in a Black-Scholes world this is the value of the contract

We can see this by simulating the diffusion in a Black-Scholes framework and observing the convergence as above

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# --- 1. DATA GENERATION & MATH SETUP ---
# ==========================================
np.random.seed(42)

N_trades = 3000
n_players = 5
start_wealth = 500  # Increased so paths survive long enough to look interesting

# Options Math (Simplified Digital/Binary Call Option)
# 30% chance to expire ITM (Payoff = $10)
# 70% chance to expire OTM (Payoff = $0)
p_itm = 0.30
p_otm = 0.70
payoff_itm = 10.0
payoff_otm = 0.0

fair_value = (p_itm * payoff_itm) + (p_otm * payoff_otm)  # $3.00
premium_paid = 3.20  # Overpaying due to implied vol / spread
edge_per_trade = fair_value - premium_paid  # -0.20

t_steps = np.arange(0, N_trades + 1)

# A. Fair Game Paths (Paying exactly $3.00, EV = 0)
# Profit if ITM = +$7, if OTM = -$3
fair_steps = np.random.choice([payoff_itm - fair_value, payoff_otm - fair_value], 
                              p=[p_itm, p_otm], size=(n_players, N_trades))
fair_paths = np.zeros((n_players, N_trades + 1))
fair_paths[:, 0] = start_wealth
for i in range(n_players):
    fair_paths[i, 1:] = start_wealth + np.cumsum(fair_steps[i])

# B. Actual Trading Paths (Paying $3.20, EV = -0.20, absorbing state at $0)
# Profit if ITM = +$6.80, if OTM = -$3.20
actual_steps = np.random.choice([payoff_itm - premium_paid, payoff_otm - premium_paid], 
                                p=[p_itm, p_otm], size=(n_players, N_trades))
actual_paths = np.zeros((n_players, N_trades + 1))
actual_paths[:, 0] = start_wealth
for i in range(n_players):
    for t in range(1, N_trades + 1):
        if actual_paths[i, t - 1] <= 0:
            actual_paths[i, t] = 0  # Absorbed (Account Blown Up)
        else:
            actual_paths[i, t] = actual_paths[i, t - 1] + actual_steps[i, t - 1]

# C. Global Market Expiry Sequence (For Bar Chart & LLN)
# 1 = ITM, 0 = OTM
global_draws = np.random.choice([1, 0], p=[p_itm, p_otm], size=N_trades)
global_payoffs = np.where(global_draws == 1, payoff_itm, payoff_otm)

# Law of Large Numbers Array (Tracking average realized payoff)
lln = np.zeros(N_trades + 1)
lln[1:] = np.cumsum(global_payoffs) / np.arange(1, N_trades + 1)

# Bar chart settings
bar_x = ['In-The-Money (ITM)', 'Out-of-The-Money (OTM)']
bar_y = [p_itm, p_otm]

def get_bar_colors(draw_val):
    # Dimmed colors by default (Green for ITM, Gray for OTM)
    colors = ['rgba(0, 255, 0, 0.15)', 'rgba(150, 150, 150, 0.15)']
    if draw_val is None:
        return colors
    if draw_val == 1:
        colors[0] = 'rgba(0, 255, 0, 1.0)'      # Active ITM
    else:
        colors[1] = 'rgba(200, 200, 200, 1.0)'  # Active OTM
    return colors

# Path Colors (Varying Opacities)
fair_colors = [f'rgba(255, 255, 255, {0.4 + 0.15 * i})' for i in range(n_players)]
actual_colors = [f'rgba(255, 0, 0, {0.4 + 0.15 * i})' for i in range(n_players)]

# ==========================================
# --- 2. FIGURE & TRACE INITIALIZATION ---
# ==========================================
fig = make_subplots(
    rows=2, cols=2, 
    vertical_spacing=0.15, horizontal_spacing=0.10,
    subplot_titles=[
        "<b>Fair Options Pricing (EV = 0)</b>", 
        "<b>Option Expiry Probability & Draw</b>", 
        "<b>Overpaying Premium (EV < 0)</b>", 
        "<b>Cumulative Average Realized Payoff</b>"
    ]
)

# 0..4: Top-Left Traces (Fair Paths - White)
for i in range(n_players):
    fig.add_trace(go.Scatter(x=[0], y=[start_wealth], mode='lines', line=dict(color=fair_colors[i], width=2)), row=1, col=1)

# 5: Top-Right Trace (Bar Chart)
fig.add_trace(go.Bar(x=bar_x, y=bar_y, marker_color=get_bar_colors(None)), row=1, col=2)

# 6..10: Bottom-Left Traces (Actual Paths - Red)
for i in range(n_players):
    fig.add_trace(go.Scatter(x=[0], y=[start_wealth], mode='lines', line=dict(color=actual_colors[i], width=2)), row=2, col=1)

# 11: Bottom-Right Trace (LLN)
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines', line=dict(color='#00ffff', width=2)), row=2, col=2)

# Static H-Lines for Law of Large Numbers chart
# Theoretical Fair Value
fig.add_shape(type="line", x0=0, y0=fair_value, x1=N_trades, y1=fair_value, 
              line=dict(color="white", width=2, dash="dot"), row=2, col=2)
# Premium Paid (Negative Edge)
fig.add_shape(type="line", x0=0, y0=premium_paid, x1=N_trades, y1=premium_paid, 
              line=dict(color="red", width=2, dash="dash"), row=2, col=2)

# ==========================================
# --- 3. ANIMATION FRAMES & SLIDER ---
# ==========================================
frames = []
step_size = max(1, N_trades // 100) # ~100 frames
frame_indices = list(range(1, N_trades + 1, step_size))
if frame_indices[-1] != N_trades:
    frame_indices.append(N_trades)

slider_steps = []

for k in frame_indices:
    frame_data = []
    
    # Update TL (Fair Paths)
    for i in range(n_players):
        frame_data.append(go.Scatter(x=t_steps[:k+1], y=fair_paths[i, :k+1]))
        
    # Update TR (Active Draw highlight)
    current_draw = global_draws[k-1]
    frame_data.append(go.Bar(marker_color=get_bar_colors(current_draw)))
    
    # Update BL (Actual Paths)
    for i in range(n_players):
        frame_data.append(go.Scatter(x=t_steps[:k+1], y=actual_paths[i, :k+1]))
        
    # Update BR (LLN)
    frame_data.append(go.Scatter(x=t_steps[1:k+1], y=lln[1:k+1]))
    
    frame_name = f"step{k}"
    frames.append(go.Frame(data=frame_data, name=frame_name))
    
    slider_steps.append({
        "args": [[frame_name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate", "transition": {"duration": 0}}],
        "label": f"{k}",
        "method": "animate"
    })

fig.frames = frames

# ==========================================
# --- 4. LAYOUT & ANNOTATIONS ---
# ==========================================
fig.update_layout(
    height=850, width=1100,
    template="plotly_dark",
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    showlegend=False,
    sliders=[{
        "active": 0,
        "yanchor": "top", "xanchor": "left",
        "currentvalue": {"font": {"size": 14, "color": "white"}, "prefix": "Options Traded: ", "visible": True, "xanchor": "right"},
        "pad": {"b": 10, "t": 60},
        "len": 0.82, "x": 0.18, "y": -0.10,
        "steps": slider_steps
    }],
    updatemenus=[{
        "type": "buttons",
        "showactive": False,
        "x": 0.0, "y": -0.10, 
        "xanchor": "left", "yanchor": "top",
        "direction": "left",
        "bgcolor": "rgba(0,0,0,0)",
        "font": {"color": "white", "size": 12}, 
        "buttons": [{
            "label": "▶ Play Simulation",
            "method": "animate",
            "args": [None, {"frame": {"duration": 40, "redraw": True}, "fromcurrent": True, "mode": "immediate"}]
        }],
        "bordercolor": "white", 
        "borderwidth": 1
    }]
)

# Axis Boundaries
y_max_fair = np.max(fair_paths) * 1.05
y_min_fair = min(np.min(fair_paths) * 1.05, -50) 

fig.update_xaxes(title_text='Trades', range=[0, N_trades], row=1, col=1)
fig.update_yaxes(title_text='Account Balance ($)', range=[y_min_fair, y_max_fair], row=1, col=1)

fig.update_yaxes(title_text='Probability', range=[0, 0.8], tickformat=".0%", row=1, col=2)

fig.update_xaxes(title_text='Trades', range=[0, N_trades], row=2, col=1)
fig.update_yaxes(title_text='Account Balance ($)', range=[0, start_wealth * 1.5], row=2, col=1)

fig.update_xaxes(title_text='Trades', range=[0, N_trades], row=2, col=2)
# Center LLN chart around the $3.00 fair value
fig.update_yaxes(title_text='Avg Payoff ($)', range=[2.0, 4.0], row=2, col=2)

# Information Annotations
fig.add_annotation(text="<b>Hedged / Fair Price</b><br>EV = $0.00", row=1, col=1, x=0.02, y=0.95, xref="x domain", yref="y domain", showarrow=False, font=dict(color="white"), bgcolor="rgba(0,0,0,0.6)", bordercolor="white", borderwidth=1)
fig.add_annotation(text="<b>Paying Premium</b><br>Overpaying by $0.20", row=2, col=1, x=0.02, y=0.95, xref="x domain", yref="y domain", showarrow=False, font=dict(color="red"), bgcolor="rgba(0,0,0,0.6)", bordercolor="red", borderwidth=1)
fig.add_annotation(text="<b>Market Edge</b><br>Red Line: Premium Paid<br>White Line: Fair EV", row=2, col=2, x=0.98, y=0.95, xref="x domain", yref="y domain", showarrow=False, font=dict(color="white"), bgcolor="rgba(0,0,0,0.6)", bordercolor="white", borderwidth=1)

fig.show()

###### ______________________________________________________________________________________________________________________________________

##### 📈 Trading with an Edge

In practice, the Black-Scholes assumptions don't hold - no kidding, all models are wrong but some are useful and this one is

If we calibrate risk-neutral dynamics to a market surface and price an instrument we get our risk-neutral expected price

 $$
 I = e^{-rT} \;\mathbb{E}^{\mathbb{Q}} [\, f(S_T) \,] \quad \text{ for some payoff } f(S_T)
 $$

 
 $$
 \text{Trade prices:} \qquad I \pm \frac{\text{spread}}{2} \implies \text{EV} = \text{spread} \quad \text{(drift in wealth)}
 $$
 
In other words, if we apply a hedging argument to a model we find the fair price, if we trade a spread we can accumulate wealth

This is what market makers do, they just have to hedge reasonably well to not decimate the collection of a spread

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# --- 1. DATA GENERATION & MATH SETUP ---
# ==========================================
np.random.seed(42)

N_trades = 10000
n_players = 5
start_wealth = 0

# Market Making Math (Options Hedging)
option_fair_value = 3.00  # Theoretical Mid Price
half_spread = 0.10        # Spread / 2 (The MM's theoretical edge per trade)

ask_price = option_fair_value + half_spread  # $3.10 (Client buys, MM sells)
bid_price = option_fair_value - half_spread  # $2.90 (Client sells, MM buys)

t_steps = np.arange(0, N_trades + 1)

# A. Market Maker Wealth Paths (Accumulating Spread/2 with Hedging Noise)
# In reality, delta hedging isn't perfect. We add variance to represent 
# slippage, gamma bleed, and inventory risk.
hedging_noise_std = 2.5 
realized_edge = half_spread + np.random.normal(0, hedging_noise_std, size=(n_players, N_trades))

mm_paths = np.zeros((n_players, N_trades + 1))
mm_paths[:, 0] = start_wealth
for i in range(n_players):
    mm_paths[i, 1:] = start_wealth + np.cumsum(realized_edge[i])

# Theoretical Linear Trend (Perfect realization of spread/2 per trade)
theoretical_trend = start_wealth + (t_steps * half_spread)

# B. Global Market Trade Sequence (For LLN Chart)
# Clients randomly cross the spread
global_trades = np.random.choice([ask_price, bid_price], size=N_trades)

# Law of Large Numbers Array (Tracking cumulative average transaction price)
lln_price = np.zeros(N_trades + 1)
lln_price[1:] = np.cumsum(global_trades) / np.arange(1, N_trades + 1)

# Path Colors (Green to symbolize accumulating edge)
mm_colors = [f'rgba(0, 255, 100, {0.4 + 0.15 * i})' for i in range(n_players)]

# ==========================================
# --- 2. FIGURE & TRACE INITIALIZATION ---
# ==========================================
fig = make_subplots(
    rows=1, cols=2, 
    horizontal_spacing=0.10,
    subplot_titles=[
        "<b>Market Maker Hedged P&L (Accumulating Spread/2)</b>", 
        "<b>Client Execution Prices & Mid Convergence</b>"
    ]
)

# Left Traces: Theoretical Trend Line
fig.add_trace(go.Scatter(x=[0], y=[start_wealth], mode='lines', 
                         line=dict(color='yellow', width=3, dash='dash'), 
                         name='Theoretical Edge'), row=1, col=1)

# Left Traces: Market Maker Hedged Paths
for i in range(n_players):
    fig.add_trace(go.Scatter(x=[0], y=[start_wealth], mode='lines', 
                             line=dict(color=mm_colors[i], width=1.5)), row=1, col=1)

# Right Trace: LLN Convergence Line
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines', 
                         line=dict(color='#00ffff', width=3)), row=1, col=2)

# Right Trace: Current Trade Marker (Blinking Dot)
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers', 
                         marker=dict(color='white', size=10, symbol='star')), row=1, col=2)

# Static H-Lines for Right Chart (Bid, Ask, Mid)
fig.add_shape(type="line", x0=0, y0=ask_price, x1=N_trades, y1=ask_price, 
              line=dict(color="#ff39b3", width=2, dash="dash"), row=1, col=2) # Ask
fig.add_shape(type="line", x0=0, y0=bid_price, x1=N_trades, y1=bid_price, 
              line=dict(color="#ff39b3", width=2, dash="dash"), row=1, col=2) # Bid
fig.add_shape(type="line", x0=0, y0=option_fair_value, x1=N_trades, y1=option_fair_value, 
              line=dict(color="white", width=2, dash="dot"), row=1, col=2)    # Mid

# Annotations for the H-Lines
fig.add_annotation(x=N_trades * 0.85, y=ask_price + 0.015, text="Offer Paid ($3.10)", showarrow=False, font=dict(color="#ff39b3", size=10), row=1, col=2)
fig.add_annotation(x=N_trades * 0.85, y=bid_price - 0.015, text="Bid Hit ($2.90)", showarrow=False, font=dict(color="#ff39b3", size=10), row=1, col=2)
fig.add_annotation(x=N_trades * 0.85, y=option_fair_value + 0.015, text="Option Fair Value ($3.00)", showarrow=False, font=dict(color="white", size=10), row=1, col=2)

# ==========================================
# --- 3. ANIMATION FRAMES & SLIDER ---
# ==========================================
frames = []
step_size = max(1, N_trades // 100)
frame_indices = list(range(1, N_trades + 1, step_size))
if frame_indices[-1] != N_trades:
    frame_indices.append(N_trades)

slider_steps = []

for k in frame_indices:
    frame_data = []
    
    # Update Left: Trend Line
    frame_data.append(go.Scatter(x=t_steps[:k+1], y=theoretical_trend[:k+1]))
    
    # Update Left: MM Paths
    for i in range(n_players):
        frame_data.append(go.Scatter(x=t_steps[:k+1], y=mm_paths[i, :k+1]))
        
    # Update Right: LLN Line
    frame_data.append(go.Scatter(x=t_steps[1:k+1], y=lln_price[1:k+1]))
    
    # Update Right: Current Trade Marker
    current_trade_price = global_trades[k-1]
    frame_data.append(go.Scatter(x=[k], y=[current_trade_price]))
    
    frame_name = f"step{k}"
    frames.append(go.Frame(data=frame_data, name=frame_name))
    
    slider_steps.append({
        "args": [[frame_name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate", "transition": {"duration": 0}}],
        "label": f"{k}",
        "method": "animate"
    })

fig.frames = frames

# ==========================================
# --- 4. LAYOUT & CONFIGURATION ---
# ==========================================
fig.update_layout(
    height=600, width=1100,
    template="plotly_dark",
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    showlegend=False,
    sliders=[{
        "active": 0,
        "yanchor": "top", "xanchor": "left",
        "currentvalue": {"font": {"size": 14, "color": "white"}, "prefix": "Trades Executed: ", "visible": True, "xanchor": "right"},
        "pad": {"b": 10, "t": 60},
        "len": 0.82, "x": 0.18, "y": -0.20,
        "steps": slider_steps
    }],
    updatemenus=[{
        "type": "buttons",
        "showactive": False,
        "x": 0.0, "y": -0.20, 
        "xanchor": "left", "yanchor": "top",
        "direction": "left",
        "bgcolor": "rgba(0,0,0,0)",
        "font": {"color": "white", "size": 12}, 
        "buttons": [{
            "label": "▶ Play Simulation",
            "method": "animate",
            "args": [None, {"frame": {"duration": 40, "redraw": True}, "fromcurrent": True, "mode": "immediate"}]
        }],
        "bordercolor": "white", 
        "borderwidth": 1
    }]
)

# Axis Boundaries
y_max_mm = max(np.max(mm_paths), np.max(theoretical_trend)) * 1.05
y_min_mm = min(np.min(mm_paths) * 1.05, -200) # Buffer for initial drawdowns

fig.update_xaxes(title_text='Option Trades', range=[0, N_trades], row=1, col=1)
fig.update_yaxes(title_text='Cumulative P&L ($)', range=[y_min_mm, y_max_mm], row=1, col=1)

fig.update_xaxes(title_text='Option Trades', range=[0, N_trades], row=1, col=2)
# Center exactly around the $3.00 option price
fig.update_yaxes(title_text='Execution Price ($)', range=[option_fair_value - 0.25, option_fair_value + 0.25], row=1, col=2)

fig.show()

Yes, the world doesn't operate by model dynamics...but it's certainly good enough

*It's amazing how wrong we can be and still make money*

---

#### 2.) 🎯 What is the Optimal Decision?

Let's consider an extreme example...

You are managing capital and have to allocate 100% to one of
- AAPL
- NVDA

Which do you choose?  Should you base it on...
- Historical returns
- Fundamentals
- News sentiment

Which is correct?  This is amorphous, like which pitch to swing at in baseball 

Optimal action requires model informed decision making, knowledge and experience

###### ______________________________________________________________________________________________________________________________________

##### ✅ Which is correct?  We will never know.

We see one sample path, maybe you had a team of quants who chose NVDA, AAPL but it returns -10%, 20%

Some guy ran analysis in his basement and chose NVDA, AAPL and it's up 20%, -10%

Whose correct?  We will never know, we walk one sample path, and the outcome doesn't dictate the quality of the decision making

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# --- 1. DATA GENERATION & MATH SETUP ---
# ==========================================
np.random.seed(500) 

N_days = 252  # 1 Trading Year
t_steps = np.arange(0, N_days + 1)
n_background_paths = 10

# Helper function: Brownian Bridge to force a path to end at a specific return
def generate_brownian_bridge(target_return, vol=0.30):
    dt = 1 / N_days
    # Generate standard random walk
    W = np.zeros(N_days + 1)
    for i in range(1, N_days + 1):
        W[i] = W[i-1] + np.random.normal(0, vol * np.sqrt(dt))
    
    # Force the endpoint using a bridge
    t_frac = t_steps / N_days
    bridge = W - t_frac * (W[-1] - target_return)
    return bridge

# Helper for background paths (Geometric Brownian Motion)
def generate_gbm_paths(n, mu, sigma):
    dt = 1 / N_days
    st_noise = np.random.normal(0, sigma * np.sqrt(dt), size=(n, N_days))
    drift = (mu - 0.5 * sigma**2) * dt
    cumulative_returns = np.exp(np.cumsum(drift + st_noise, axis=1)) - 1
    paths = np.hstack([np.zeros((n, 1)), cumulative_returns])
    return paths

# --- NVDA DATA (LEFT CHART) - THE LUCKY GAMBLE ---
# NEGATIVE expected return (-0.25), High volatility (0.40)
nvda_mu, nvda_sig = -0.25, 0.40
# One green path forcing a lucky +25%
nvda_realized_green = generate_brownian_bridge(0.25, vol=nvda_sig)
# 10 background paths (mostly drifting down)
nvda_multiverse = generate_gbm_paths(n_background_paths, nvda_mu, nvda_sig)

# --- AAPL DATA (RIGHT CHART) - THE UNLUCKY QUANT ---
# POSITIVE expected return (+0.25), Moderate volatility (0.25)
aapl_mu, aapl_sig = 0.25, 0.25
# One blue path forcing an unlucky -10%
aapl_realized_blue = generate_brownian_bridge(-0.10, vol=aapl_sig)
# 10 background paths (mostly drifting up)
aapl_multiverse = generate_gbm_paths(n_background_paths, aapl_mu, aapl_sig)

# ==========================================
# --- 2. FIGURE & TRACE INITIALIZATION ---
# ==========================================
fig = make_subplots(
    rows=1, cols=2, 
    horizontal_spacing=0.10,
    subplot_titles=[
        "<b>NVDA: Bad Process, Lucky Outcome</b>", 
        "<b>AAPL: Good Process, Unlucky Outcome</b>"
    ]
)

# Stylistic choices
bg_colors = [f'rgba(255, 255, 255, {0.05 + 0.015*i})' for i in range(n_background_paths)]
realized_green_col = 'rgba(0, 255, 100, 1.0)' 
realized_blue_col = 'rgba(0, 150, 255, 1.0)'  

# --- Add NVDA Traces (Left) ---
for i in range(n_background_paths):
    fig.add_trace(go.Scatter(x=[0], y=[0], mode='lines', line=dict(color=bg_colors[i], width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=[0], y=[0], mode='lines', line=dict(color=realized_green_col, width=3)), row=1, col=1)

# --- Add AAPL Traces (Right) ---
for i in range(n_background_paths):
    fig.add_trace(go.Scatter(x=[0], y=[0], mode='lines', line=dict(color=bg_colors[i], width=1)), row=1, col=2)
fig.add_trace(go.Scatter(x=[0], y=[0], mode='lines', line=dict(color=realized_blue_col, width=3)), row=1, col=2)

# ==========================================
# --- 3. ANIMATION FRAMES & SLIDER ---
# ==========================================
frames = []
step_size = max(1, N_days // 100)
frame_indices = list(range(1, N_days + 1, step_size))
if frame_indices[-1] != N_days:
    frame_indices.append(N_days)

slider_steps = []

for k in frame_indices:
    frame_data = []
    
    # Update NVDA
    for i in range(n_background_paths):
        frame_data.append(go.Scatter(x=t_steps[:k+1], y=nvda_multiverse[i, :k+1]))
    frame_data.append(go.Scatter(x=t_steps[:k+1], y=nvda_realized_green[:k+1]))
    
    # Update AAPL
    for i in range(n_background_paths):
        frame_data.append(go.Scatter(x=t_steps[:k+1], y=aapl_multiverse[i, :k+1]))
    frame_data.append(go.Scatter(x=t_steps[:k+1], y=aapl_realized_blue[:k+1]))
    
    frame_name = f"step{k}"
    frames.append(go.Frame(data=frame_data, name=frame_name))
    
    slider_steps.append({
        "args": [[frame_name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate", "transition": {"duration": 0}}],
        "label": f"{k}",
        "method": "animate"
    })

fig.frames = frames

# ==========================================
# --- 4. LAYOUT: DARK THEME & BUTTONS ---
# ==========================================
fig.update_layout(
    height=600, width=1100,
    template="plotly_dark",
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    showlegend=False,
    sliders=[{
        "active": 0,
        "yanchor": "top", "xanchor": "left",
        "currentvalue": {"font": {"size": 14, "color": "white"}, "prefix": "Trading Days: ", "visible": True, "xanchor": "right"},
        "pad": {"b": 10, "t": 60},
        "len": 0.82, "x": 0.18, "y": -0.20,
        "steps": slider_steps
    }],
    updatemenus=[{
        "type": "buttons",
        "showactive": False,
        "x": 0.0, "y": -0.20, 
        "xanchor": "left", "yanchor": "top",
        "direction": "left",
        "bgcolor": "rgba(0,0,0,0)",
        "font": {"color": "white", "size": 12}, 
        "buttons": [{
            "label": "▶ Play Simulation",
            "method": "animate",
            "args": [None, {"frame": {"duration": 40, "redraw": True}, "fromcurrent": True, "mode": "immediate"}]
        }],
        "bordercolor": "white", 
        "borderwidth": 1
    }]
)

# Axis Boundaries & Formatting
y_limit = 1.0  # Capturing -100% to +100%

for col in [1, 2]:
    fig.update_xaxes(title_text='Trading Days', range=[0, N_days], row=1, col=col)
    fig.update_yaxes(title_text='Cumulative Return', tickformat=".0%", range=[-y_limit, y_limit], row=1, col=col)
    fig.add_shape(type="line", x0=0, y0=0, x1=N_days, y1=0, line=dict(color="rgba(255,255,255,0.3)", width=1, dash="dot"), row=1, col=col)

fig.show()

###### ______________________________________________________________________________________________________________________________________

##### 📈 Trading with an Edge

The objective of course is to be *correct* more often than not, we can't **result** with one decision

However, we can observe a composite of decisions made by an individual or a firm to see if they are acting optimally over time

There is no gaurentee they will continue, think like a batting average

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# --- 1. DATA GENERATION & PROCESS SIM ---
# ==========================================
np.random.seed(42)

N_days = 252 * 2  # 2 Trading Years for LLN to begin to emerge clearly
t_steps = np.arange(0, N_days + 1)
n_bg_paths = 15

# Common formatting: Equity curve represents cumulative return starting at 0%

# A. Basement Guy (Negative Edge)
# Parameters: EV < 0, High Variance/Luck dependency
bg_mu_yearly, bg_sigma_yearly = -0.20, 0.35 # Drift -20%, Vol 35%
dt = 1/252

# Generate many paths (1 main highlighted + background ones)
bg_paths_raw = np.zeros((n_bg_paths + 1, N_days + 1))
for i in range(n_bg_paths + 1):
    # Arithmetic random walk simulation of equity
    daily_returns = np.random.normal(bg_mu_yearly * dt, bg_sigma_yearly * np.sqrt(dt), N_days)
    bg_paths_raw[i, 1:] = np.cumsum(daily_returns)

# Select one path to be the main highlighted path, ensuring it illustrates the trend
# though randomness will usually handle this with enough paths.
bg_main_path = bg_paths_raw[-1, :]
bg_bg_paths = bg_paths_raw[:-1, :]


# B. Quant Firm (Positive Edge)
# Parameters: EV > 0, Moderate Variance
qf_mu_yearly, qf_sigma_yearly = 0.25, 0.20 # Drift +25%, Vol 20%

qf_paths_raw = np.zeros((n_bg_paths + 1, N_days + 1))
for i in range(n_bg_paths + 1):
    daily_returns = np.random.normal(qf_mu_yearly * dt, qf_sigma_yearly * np.sqrt(dt), N_days)
    qf_paths_raw[i, 1:] = np.cumsum(daily_returns)

qf_main_path = qf_paths_raw[-1, :]
qf_bg_paths = qf_paths_raw[:-1, :]

# ==========================================
# --- 2. FIGURE & TRACE INITIALIZATION ---
# ==========================================
fig = make_subplots(
    rows=1, cols=2, 
    horizontal_spacing=0.10,
    subplot_titles=[
        "<b>Basement Guy Process (Negative Edge)</b>", 
        "<b>Quant Firm Process (Positive Edge)</b>"
    ]
)

# Stylistic options: Matching image_0 style
# Background paths use varying opacities for depth distribution visualization
# Left: Neon Pink (`#ff39b3`). Right: Neon Cyan (`#00ffff`).
l_main_col = '#ff39b3' 
r_main_col = '#00ffff'
bg_base_col = '200, 200, 200' # Grey/White base for opacities

# Add Left Traces (0 to n_bg_paths: total n+1 traces)
for i in range(n_bg_paths):
    # Background Paths
    opacity = 0.05 + (i / n_bg_paths) * 0.15 # 5% to 20%
    fig.add_trace(go.Scatter(x=[0], y=[0], mode='lines', line=dict(color=f'rgba({bg_base_col}, {opacity})', width=1)), row=1, col=1)
# Highlighted main path on top
fig.add_trace(go.Scatter(x=[0], y=[0], mode='lines', line=dict(color=l_main_col, width=3)), row=1, col=1)

# Add Right Traces (n+1 to 2n+2: total n+1 traces)
for i in range(n_bg_paths):
    # Background Paths
    opacity = 0.05 + (i / n_bg_paths) * 0.15
    fig.add_trace(go.Scatter(x=[0], y=[0], mode='lines', line=dict(color=f'rgba({bg_base_col}, {opacity})', width=1)), row=1, col=2)
# Highlighted main path on top
fig.add_trace(go.Scatter(x=[0], y=[0], mode='lines', line=dict(color=r_main_col, width=3)), row=1, col=2)


# ==========================================
# --- 3. ANIMATION FRAMES & SLIDER ---
# ==========================================
frames = []
# Ensure smooth playback: create ~100 frames total over N days
step_size = max(1, N_days // 100) 
frame_indices = list(range(1, N_days + 1, step_size))
# Force last frame if step_size misses exactly N_days
if frame_indices[-1] != N_days:
    frame_indices.append(N_days)

slider_steps = []

for k in frame_indices:
    frame_data = []
    
    # Update Left Subplot
    for i in range(n_bg_paths):
        frame_data.append(go.Scatter(x=t_steps[:k+1], y=bg_bg_paths[i, :k+1]))
    frame_data.append(go.Scatter(x=t_steps[:k+1], y=bg_main_path[:k+1]))
    
    # Update Right Subplot
    for i in range(n_bg_paths):
        frame_data.append(go.Scatter(x=t_steps[:k+1], y=qf_bg_paths[i, :k+1]))
    frame_data.append(go.Scatter(x=t_steps[:k+1], y=qf_main_path[:k+1]))
    
    frame_name = f"step{k}"
    frames.append(go.Frame(data=frame_data, name=frame_name))
    
    # Slider configuration: Matching image_0 layout
    slider_steps.append({
        "args": [[frame_name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate", "transition": {"duration": 0}}],
        "label": f"{t_steps[k]/252:.2f}",
        "method": "animate"
    })

fig.frames = frames

# ==========================================
# --- 4. LAYOUT: DARK THEME & BUTTONS ---
# ==========================================
# Setup derived from image_0.png, using dark template and matching button/slider placement.
fig.update_layout(
    height=600, width=1000,
    template="plotly_dark",
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    showlegend=False,
    # Align play button and slider on y-axis (both at y = -0.10, matching image_0)
    sliders=[{
        "active": 0,
        "yanchor": "top", "xanchor": "left",
        "currentvalue": {"font": {"size": 14, "color": "white"}, "prefix": "Simulation Year: ", "visible": True, "xanchor": "right"},
        "pad": {"b": 10, "t": 60},
        "len": 0.82, "x": 0.18, "y": -0.10,
        "steps": slider_steps
    }],
    updatemenus=[{
        "type": "buttons",
        "showactive": False,
        "x": 0.0, "y": -0.10, 
        "xanchor": "left", "yanchor": "top",
        "direction": "left",
        "bgcolor": "rgba(0,0,0,0)", # Transparent background
        "font": {"color": "white", "size": 12}, 
        "buttons": [{
            "label": "▶ Play Simulation",
            "method": "animate",
            "args": [None, {"frame": {"duration": 40, "redraw": True}, "fromcurrent": True, "mode": "immediate"}]
        }],
        "bordercolor": "white", 
        "borderwidth": 1
    }]
)

# Axis Boundaries & Standard Zero Line
# Use symmetrical range to center around 0% for easier visual comparison.
y_limit = max(np.max(np.abs(qf_paths_raw)), np.max(np.abs(bg_paths_raw))) * 1.05

for col in [1, 2]:
    fig.update_xaxes(title_text='Trading Days', range=[0, N_days], row=1, col=col)
    fig.update_yaxes(title_text='Cumulative Return', tickformat=".0%", range=[-y_limit, y_limit], row=1, col=col)
    # Add dotted zero line to emphasize trend vs realization crossing zero
    fig.add_shape(type="line", x0=0, y0=0, x1=N_days, y1=0, line=dict(color="rgba(255,255,255,0.3)", width=1, dash="dot"), row=1, col=col)

fig.show()

---

#### 3.) 💭 Closing Thoughts and Future Topics

**TL;DW Executive Summary**
  - Pricing in quantitative finance is fundamentally about modeling uncertainty: we use mathematical models to assign a fair value to assets, derivatives, and portfolios based on the probabilities of various future scenarios. 
  - Techniques like risk-neutral valuation, Black-Scholes, and Monte Carlo simulations are used to estimate fair prices, all acknowledging unavoidable market randomness.
  - In quantitative finance, choosing the "correct" action is uncertain; we only see one outcome path.
  - Decision quality can't be judged solely by results, as good choices can lead to bad outcomes and vice versa.
  - Optimal trading decisions require models, experience, and knowledge—not just luck or intuition.
  - Being "right" is a matter of probabilities and consistent good decision-making over time.
  - Monitoring performance over many decisions (like a batting average) gives a better sense of skill than any single result.


**Future Topics**

Technical Videos and Other Discussions

- Hawkes Processes
- Merton Jump Diffusion Model (and Characteristic Function Pricing, Carr-Madan 1999)
- Market-Making Models and Simulation (Stoikov-Avellaneda)
- Projects that Made me a Quant
- My First Year as a Quant
- Kalman Filter for Quant Finance
- Why Hedge Funds are Actually Secretive
- Non-Markovian Models (fractional Brownian motion, Volterra Process)
- Top 3 Uses of Linear Algebra for Quant Finance
- Girsanov's Change of Measure
- Rough Path Theory, Applications of Path Signatures
- Sig-Vol Model, Calibration, and Pricing

[Ideas for Interactive Brokers Apps and Tutorials](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

- How to Backtest a Trading Strategy with Interactive Brokers
- Algorithmic Volatility Trading System

---

####  $\text{Copyright © 2026 Quant Guild} \quad \quad \quad \quad \text{Author: Roman Paolucci}$